Sound files from https://github.com/Jakobovski/free-spoken-digit-dataset/tree/master

Imports

In [ ]:
# for tensors, relu, matrix multiplication, back-propagation, etc
import torch
import torch.nn.functional as F
# for file io
import os
# for converting audio files to MFCC vectors
import librosa
# for padding and numpy array datastructure
import numpy as np

from sklearn.model_selection import train_test_split

Google Drive mounting

In [ ]:
# where I saved my files
from google.colab import drive
drive.mount("/content/drive", force_remount = True)

Mounted at /content/drive


Universal constants

In [ ]:
custom = False

# the folder with the sound files I made
if custom:
  data_dir = "/content/drive/My Drive/Grade 12/AI/Numbers/CustomDataset"
# the folder with the sound files I got from GitHub
else:
  data_dir = "/content/drive/My Drive/Grade 12/AI/Numbers/300Dataset"

n_mfcc = 13 # number of MFCC coefficients
max_len = 100 # number of time steps (change to match ideal sound file length)
num_classes = 10 # 0-9
input_size = n_mfcc * max_len # size of flat tensors

Class for MFCC conversion

In [ ]:
class Converter:
  def __init__(self, n_mfcc):
    self._n_mfcc = n_mfcc

  def set_path(self, path):
    # Load audio
    self._audio, self._sr = librosa.load(path, sr=None)

  def convert(self, max_len):
    # turn sound files into vectors
    self._mfcc = librosa.feature.mfcc(y=self._audio, sr=self._sr, n_mfcc=self._n_mfcc)
    self.pad(max_len)
    return self.get_flat()

  def pad(self, max_len):
    # pad or truncate vectors to make them all the same length, regardless of audio file length
    length = self._mfcc.shape[1]
    if length < max_len:
        pad_size = max_len - length
        self._mfcc = np.pad(self._mfcc, ((0, 0), (0, pad_size)))
    else:
        self._mfcc = self._mfcc[:, :max_len]

  def get_flat(self):
    # flattened from 2D to 1D vector
    return self._mfcc.flatten()

Convert sound files to MFCC vectors & store them

In [ ]:
X = []
y = []

converter = Converter(n_mfcc)

# loop through audio files in each number's folder in the directory containing training set
for label in range(num_classes):
    folder = os.path.join(data_dir, str(label))

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        converter.set_path(path)
        mfcc = converter.convert(max_len)

        X.append(mfcc)
        y.append(label)

/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1891
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1565
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1720
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2007
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1933
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2016
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py

Split training & testing sets, convert to tensors

In [ ]:
# Convert to numpy array so scikit can work with them
X = np.array(X)
y = np.array(y)

# Normalize MFCC vectors (for example, make two "zero"s of different volume more similar)
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

# Split training & testing; "stratify=y" ensures that there is an equal number of each class in the sets; for example, 20% of zeroes and 20% of ones, as opposed to a random 20% of all numbers
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Convert from arrays to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Normalize datatypes
X_train = X_train.float()
X_test = X_test.float()
y_train = y_train.long()
y_test = y_test.long()

print("X training shape:", X_train.shape)
print("X testing shape:", X_test.shape)
print("y training shape:", y_train.shape)
print("y testing shape:", y_test.shape)

X training shape: torch.Size([2400, 1300])
X testing shape: torch.Size([600, 1300])
y training shape: torch.Size([2400])
y testing shape: torch.Size([600])


Neural net class

In [ ]:
class FCNN:
  def __init__(self, input_size, num_classes, hidden_sizes = [64], learning_rate = 0.1):
    self._input_size = input_size
    self._num_classes = num_classes
    self._hidden_sizes = hidden_sizes
    self._learning_rate = learning_rate

    self._layer_sizes = [self._input_size] + self._hidden_sizes + [self._num_classes]

    self._weights = []
    self._biases = []

    #make list of hidden layers (input size > hidden_sizes[0], hidden_sizes[0] > ..., ... > hidden_sizes[-1], hidden_sizes[-1] > num classes)
    for i in range(len(self._layer_sizes) - 1):
      w = torch.randn(self._layer_sizes[i], self._layer_sizes[i+1]) * 0.01 # with small dataset, start with smaller weights (so multiply by 0.01)
      b = torch.zeros(self._layer_sizes[i+1], requires_grad=True) # don't really need biases (I think)

      w.requires_grad_()

      self._weights.append(w)
      self._biases.append(b)

  def classify(self, x):
    out = x

    # every layer except the last one with ReLU (non-negative-ization)
    for i in range(len(self._weights) - 1):
      out = torch.relu(torch.matmul(out, self._weights[i]) + self._biases[i])

    # last layer without ReLU
    out = torch.matmul(out, self._weights[-1]) + self._biases[-1]
    return out

  def train(self, X_train, y_train):
    # Training Loop
    for epoch in range(10000):
      preds = self.classify(X_train)
      # cross_entropy instead of mean squared error because I'm doing classification instead of regression
      loss = F.cross_entropy(preds, y_train)

      loss.backward()

      # back propagation
      with torch.no_grad():
        for w, b in zip(self._weights, self._biases):
          w -= self._learning_rate * w.grad
          b -= self._learning_rate * b.grad
          w.grad.zero_()
          b.grad.zero_()

  def test(self, X_test, y_test):
    with torch.no_grad():
      out = self.classify(X_test)

      predictions = torch.argmax(out, dim = 1)

      accuracy = (predictions == y_test).float().mean()

    return accuracy.item()

Train and test with existing dataset

In [ ]:
fcnn = FCNN(input_size, num_classes)
fcnn.train(X_train, y_train)
accuracy = fcnn.test(X_test, y_test)
print("Accuracy: ", accuracy)

Accuracy:  0.9766666889190674


Predict a singular sound

In [ ]:
single_folder_path = "/content/drive/My Drive/Grade 12/AI/Numbers/Single"

for file in os.listdir(single_folder_path):
  path = os.path.join(single_folder_path, file)

converter.set_path(path)
mfcc = converter.convert(max_len)

X = (mfcc - mfcc.mean()) / mfcc.std()
X = np.array(X)
X = torch.tensor(X, dtype=torch.float32)

pred = torch.argmax(fcnn.classify(X))
print("Predicted label: ", pred.item())

Predicted label:  5


/tmp/ipykernel_2614/2146966364.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  self._audio, self._sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
